# Part 10 — Build and train a real DDPM

_Rigorous Courses · Diffusion Models — Part 10 of 12_

**Turn the noise-prediction loss into working PyTorch and watch pure noise organize into data**

In this notebook you build every piece of a real DDPM: the eight-cluster toy dataset, the noise schedule, a sinusoidal time embedding, a small MLP noise-guesser, the Algorithm 1 training loop, and the Algorithm 2 sampling loop. You then score the samples with mode-coverage and moment checks, and close with a teaser on why naive step-skipping fails.

---

This notebook accompanies the lesson. Run cells top to bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / matplotlib ship with Colab; torch is preinstalled there too.
import math

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

rng = np.random.default_rng(0)
torch.manual_seed(0)

print(f"torch version: {torch.__version__} (CPU is all we need)")

## The dataset: a ring of eight clusters

We need data we can see whole. Our dataset: 4,096 points in the plane, in 8 Gaussian clusters spaced evenly around a circle of radius 4, each with spread 0.15. It has crowded regions (the clusters) and empty regions (everywhere else) — a distribution worth learning, small enough to train in seconds.

### Step 1 — Build the eight-mode ring

For each point: pick a cluster uniformly at random, place its center on the circle of radius 4, then sprinkle the point around that center with spread 0.15 (part 3's reparameterization $x = \mu + \sigma z$ in code). We check the shape, the mean radius, and the within-cluster spread.

In [ ]:
n = 4096

mode = rng.integers(0, 8, size=n)
angle = 2 * np.pi * mode / 8
centers = 4.0 * np.stack([np.cos(angle), np.sin(angle)], axis=1)
x0_np = centers + 0.15 * rng.standard_normal((n, 2))
data = torch.tensor(x0_np, dtype=torch.float32)

radius = np.linalg.norm(x0_np, axis=1)
spread0 = x0_np[mode == 0].std(axis=0).mean()

assert data.shape == (4096, 2)
assert abs(radius.mean() - 4.0) < 0.05

print(f"data shape: {tuple(data.shape)}")
print(f"mean radius: {radius.mean():.3f} (target 4.0)")
print(f"within-cluster spread (cluster 0): {spread0:.3f} (target 0.15)")

### Step 2 — Look at the data cloud

Eight tight blobs on a ring. Keep this picture in mind: a trained model must recreate it from pure noise — all 8 clusters, the right radius, the right spread.

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(x0_np[:, 0], x0_np[:, 1], s=4, alpha=0.4, color="#4ea1ff")
plt.title("Eight-mode ring dataset (n = 4096)")
plt.xlabel("first coordinate")
plt.ylabel("second coordinate")
plt.axis("equal")
plt.show()

## The noise schedule (part 6's code, now in torch)

Same recipe as part 6: $T = 200$ steps, noise doses $\beta_t$ sliding linearly from $10^{-4}$ to $0.02$, survival factors $\alpha_t = 1 - \beta_t$, and the cumulative survival $\bar\alpha_t = \prod_{s \le t} \alpha_s$ — the fraction of the original signal surviving to step $t$.

### Step 3 — Build betas, alphas, abar as torch tensors

We precompute $\sqrt{\bar\alpha_t}$ and $\sqrt{1-\bar\alpha_t}$ once — the training loop looks them up thousands of times. Note the endpoint: $\bar\alpha_T \approx 0.13$, so the compounded stretch during sampling will be $1/\sqrt{\bar\alpha_T} \approx 2.75$.

In [ ]:
T = 200

betas = torch.linspace(1e-4, 0.02, T)
alphas = 1.0 - betas
abar = torch.cumprod(alphas, dim=0)

sqrt_abar = torch.sqrt(abar)
sqrt_1m_abar = torch.sqrt(1.0 - abar)

assert torch.all(abar[1:] < abar[:-1])
assert abar.min() > 0.0
assert abar.max() < 1.0

print(f"beta_1 = {betas[0].item():.6f}   beta_T = {betas[-1].item():.6f}")
print(f"abar_1 = {abar[0].item():.4f}   abar_T = {abar[-1].item():.4f}")
print(f"total sampling stretch 1/sqrt(abar_T) = {(1.0 / sqrt_abar[-1]).item():.2f}x")

## The sinusoidal time embedding

The network must be told $t$, but the raw integer (1 to 200) is badly scaled next to coordinates of size 4, and one number is a cramped representation of 200 related-but-distinct jobs. The fix is to read $t$ off a bank of clocks ticking at different speeds:

$$e(t) = \big(\sin(\omega_1 t), \dots, \sin(\omega_K t), \cos(\omega_1 t), \dots, \cos(\omega_K t)\big), \qquad \omega_k = 10000^{-\frac{k-1}{K-1}}$$

We use $K = 16$ frequencies, so each timestep becomes 32 numbers, every one inside $[-1, 1]$. The fast hand ($\omega_1 = 1$) wraps about 32 times over our 200 steps and separates nearby timesteps; the slow hand ($\omega_{16} = 10^{-4}$) advances only about one degree in total and separates early from late.

### Step 4 — Implement the embedding and check it by hand

Three working lines: build the frequencies, form every timestep-times-frequency angle, then read each clock twice (sine and cosine). We assert the shape, the $[-1, 1]$ range, the all-zeros/all-ones pattern at $t = 0$, and the exact values from the lesson's table.

In [ ]:
emb_dim = 32
half = emb_dim // 2

freqs = torch.exp(-math.log(10000.0) * torch.arange(half) / (half - 1))


def time_embedding(t):
    args = t[:, None].float() * freqs[None, :]
    emb = torch.cat([torch.sin(args), torch.cos(args)], dim=1)
    return emb


emb_all = time_embedding(torch.arange(T))

assert emb_all.shape == (200, 32)
assert emb_all.abs().max() <= 1.0 + 1e-6
assert torch.allclose(emb_all[0, :half], torch.zeros(half))
assert torch.allclose(emb_all[0, half:], torch.ones(half))
assert abs(emb_all[50, 0].item() - math.sin(50)) < 1e-4
assert abs(emb_all[50, half].item() - math.cos(50)) < 1e-4
assert abs(emb_all[199, half - 1].item() - math.sin(199 * 1e-4)) < 1e-4

print("t     fast sin   fast cos   slow sin   slow cos")
for t_check in [0, 50, 199]:
    row = emb_all[t_check]
    print(f"{t_check:3d}   {row[0].item():+.3f}     {row[half].item():+.3f}     {row[half - 1].item():+.3f}     {row[-1].item():+.3f}")

### Step 5 — See all 200 embeddings at once

Each row of the heatmap is one timestep's 32-number fingerprint. Look at the columns: the leftmost (fastest) columns stripe rapidly down the page — they wrap many times — while columns near each half's right edge barely change — they creep. Together the rows are unique, smooth fingerprints of $t$.

In [ ]:
plt.figure(figsize=(7, 4))
plt.imshow(emb_all.numpy(), aspect="auto", cmap="RdBu", vmin=-1, vmax=1)
plt.colorbar(label="embedding value")
plt.xlabel("embedding dimension (0-15 sines, 16-31 cosines)")
plt.ylabel("timestep t")
plt.title("Sinusoidal time embedding: fast columns wrap, slow columns creep")
plt.show()

## The network: a small MLP that outputs noise

Input: the noisy point's 2 coordinates glued to the 32-number embedding — 34 numbers. Two hidden layers of 128 units with SiLU activations. Output: 2 numbers — **the same shape as the data**, because the network's job is a noise guess per point, not a label. Data-shaped in, data-shaped out.

### Step 6 — Define the MLP and count its knobs

Hand count from the lesson: $34 \times 128 + 128 = 4480$, then $128 \times 128 + 128 = 16512$, then $128 \times 2 + 2 = 258$, total $21{,}250$ parameters. We assert the code agrees, and that a 256-point batch produces a $(256, 2)$ output.

In [ ]:
class EpsMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2 + emb_dim, 128),
            nn.SiLU(),
            nn.Linear(128, 128),
            nn.SiLU(),
            nn.Linear(128, 2),
        )

    def forward(self, x, t):
        emb = time_embedding(t)
        h = torch.cat([x, emb], dim=1)
        return self.net(h)


model = EpsMLP()
n_params = sum(p.numel() for p in model.parameters())
test_out = model(data[:256], torch.randint(1, T + 1, (256,)))

assert n_params == 21250
assert test_out.shape == (256, 2)

print(f"parameter count: {n_params} (hand count: 4480 + 16512 + 258 = 21250)")
print(f"output shape for a 256-point batch: {tuple(test_out.shape)} — same shape as the data")

## Training: Algorithm 1 with minibatches

The loss from part 9, which we now minimize for real:

$$L_{\mathrm{simple}}(\theta) = \mathbb{E}_{x_0,\, t,\, \epsilon}\, \big\|\epsilon - \epsilon_\theta(x_t, t)\big\|^2, \qquad x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1-\bar\alpha_t}\, \epsilon$$

Each training step is Algorithm 1 on a batch of 256: sample clean points, sample a timestep per point, sample fresh noise, form $x_t$ with the one-line sampler, guess the noise, and take one Adam step (learning rate $10^{-3}$) on the mean squared error. The batch loss is a Monte-Carlo estimate of the expectation above — expect it to wiggle.

### Step 7 — Train for 2,500 steps

Watch the printed loss: it should start near 1.0 (an untrained network guesses about 0, and the noise has variance 1 per coordinate), fall fast, then flatten. This takes well under a minute on a CPU.

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
losses = []

for step in range(2500):
    idx = torch.randint(0, n, (256,))
    x0 = data[idx]
    t = torch.randint(1, T + 1, (256,))
    eps = torch.randn(256, 2)
    xt = sqrt_abar[t - 1][:, None] * x0 + sqrt_1m_abar[t - 1][:, None] * eps
    eps_pred = model(xt, t)
    loss = ((eps - eps_pred) ** 2).mean()
    opt.zero_grad()
    loss.backward()
    opt.step()
    losses.append(loss.item())
    if step % 250 == 0 or step == 2499:
        print(f"step {step:4d}   batch loss = {loss.item():.3f}")

### Step 8 — Read the loss curve

Three things to verify against the lesson: the start near 1.0, the noisy-but-falling shape (each point is one random batch's Monte-Carlo estimate), and the plateau **above zero** — the irreducible floor from part 9: at tiny $t$ the added noise is microscopic next to the data's own spread, so no network can guess it, no matter how long we train.

In [ ]:
losses_arr = np.array(losses)
early = losses_arr[0]
late = losses_arr[-100:].mean()
running = np.convolve(losses_arr, np.ones(50) / 50, mode="valid")

plt.figure(figsize=(7, 4))
plt.plot(losses_arr, lw=0.5, alpha=0.5, color="#4ea1ff", label="per-batch loss")
plt.plot(np.arange(49, len(losses_arr)), running, color="#ff7b72", label="50-step running average")
plt.xlabel("training step")
plt.ylabel("batch loss")
plt.title("DDPM training loss: fast fall, noisy wiggle, floor above zero")
plt.legend()
plt.show()

assert early > 0.7
assert late < early * 0.5

print(f"first batch loss: {early:.3f} (predicted: about 1.0)")
print(f"mean of last 100 batches: {late:.3f} — well above zero, as the theory demands")

## Sampling: Algorithm 2

Training never generated anything. Generation is part 9's Algorithm 2: start 2,000 points at pure noise $x_T \sim \mathcal{N}(0, \mathbf{I})$ and walk them down all 200 learned steps with

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\, \epsilon_\theta(x_t, t)\right) + \sigma_t z, \qquad \sigma_t = \sqrt{\beta_t},\ \ z = 0 \text{ at } t = 1.$$

We save snapshots at $t = 200, 150, 100, 50, 25, 0$ to watch the walk.

### Step 9 — Walk 2,000 noise points down to data

Note the two numbers printed at the end: the starting cloud has typical radius about 1.25 (a 2-D standard Gaussian), and the finished samples should sit near radius 4 — the compounded $1/\sqrt{\alpha_t}$ stretches (2.75x total) plus the network's steering supply the growth.

In [ ]:
snapshot_ts = [200, 150, 100, 50, 25, 0]
snapshots = {}

with torch.no_grad():
    xt = torch.randn(2000, 2)
    snapshots[200] = xt.clone()
    for t in range(T, 0, -1):
        tvec = torch.full((2000,), t)
        eps_pred = model(xt, tvec)
        coef = betas[t - 1] / sqrt_1m_abar[t - 1]
        mean = (xt - coef * eps_pred) / torch.sqrt(alphas[t - 1])
        if t > 1:
            z = torch.randn_like(xt)
            xt = mean + torch.sqrt(betas[t - 1]) * z
        else:
            xt = mean
        if (t - 1) in snapshot_ts:
            snapshots[t - 1] = xt.clone()

samples = snapshots[0]

print(f"start typical radius: {snapshots[200].norm(dim=1).mean().item():.2f} (theory: 1.25)")
print(f"final mean radius:    {samples.norm(dim=1).mean().item():.2f} (target: 4.0)")

### Step 10 — Watch noise organize into the ring

Read the strip left to right: a featureless blob drifts outward, a ring forms, and the ring sharpens into eight clusters. This is the reverse of part 6's iconic noising strip — run by a network you trained two cells ago.

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(16, 3))
for ax, t_snap in zip(axes, snapshot_ts):
    pts = snapshots[t_snap].numpy()
    ax.scatter(pts[:, 0], pts[:, 1], s=2, alpha=0.4, color="#4ea1ff")
    ax.set_xlim(-6, 6)
    ax.set_ylim(-6, 6)
    ax.set_title(f"t = {t_snap}")
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle("Algorithm 2: pure noise organizing into the eight-mode ring")
plt.tight_layout()
plt.show()

### Step 11 — Score the samples: modes, moments, spread

The three checks from the lesson: assign each sample to its nearest cluster center and count (coverage — we assert at least 6 of 8 modes get a meaningful share), check the mean radius (within 15% of 4), and look at the within-cluster spread (the true mean distance to a center is about $0.15\sqrt{\pi/2} \approx 0.19$).

In [ ]:
angles_8 = 2 * math.pi * torch.arange(8) / 8
centers_8 = 4.0 * torch.stack([torch.cos(angles_8), torch.sin(angles_8)], dim=1)

d2 = torch.cdist(samples, centers_8)
nearest = d2.argmin(dim=1)
counts = torch.bincount(nearest, minlength=8)
modes_hit = int((counts >= 40).sum())
mean_radius = samples.norm(dim=1).mean().item()
dist_to_mode = d2.min(dim=1).values

assert modes_hit >= 6
assert abs(mean_radius - 4.0) / 4.0 < 0.15

print(f"samples per mode: {counts.tolist()}")
print(f"modes hit (at least 40 of 2000 samples): {modes_hit} of 8")
print(f"mean radius: {mean_radius:.3f} (target 4.0, accept within 15%)")
print(f"mean distance to nearest center: {dist_to_mode.mean().item():.3f} (true value: about 0.19)")

## Ablation teaser: naive 20-step skipping

Sampling cost 200 network calls. Could we run only 20 of the 200 updates, evenly spaced, and skip the rest? Each update removes only **one step's worth** of noise — skipping 180 updates leaves 180 steps of noise never removed. Watch it fail; part 11 builds the sampler (DDIM) that skips steps properly.

### Step 12 — Sample with 20 naively chosen steps

Same trained network, same update rule, but only 20 of the 200 timesteps. We compare the mean distance to the nearest mode center against the full sampler's — expect it to be many times worse.

In [ ]:
taus = np.linspace(T, 1, 20).round().astype(int)

with torch.no_grad():
    xt = torch.randn(2000, 2)
    for t in taus:
        tvec = torch.full((2000,), int(t))
        eps_pred = model(xt, tvec)
        coef = betas[t - 1] / sqrt_1m_abar[t - 1]
        mean = (xt - coef * eps_pred) / torch.sqrt(alphas[t - 1])
        if t > 1:
            z = torch.randn_like(xt)
            xt = mean + torch.sqrt(betas[t - 1]) * z
        else:
            xt = mean

naive = xt
naive_dist = torch.cdist(naive, centers_8).min(dim=1).values.mean().item()
full_dist = dist_to_mode.mean().item()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4.5))
ax1.scatter(samples[:, 0], samples[:, 1], s=2, alpha=0.4, color="#4ea1ff")
ax1.set_title("full 200-step sampler")
ax2.scatter(naive[:, 0], naive[:, 1], s=2, alpha=0.4, color="#ff7b72")
ax2.set_title("naive 20-step skipping")
for ax in (ax1, ax2):
    ax.set_xlim(-6, 6)
    ax.set_ylim(-6, 6)
    ax.set_xlabel("first coordinate")
ax1.set_ylabel("second coordinate")
plt.tight_layout()
plt.show()

assert naive_dist > 2 * full_dist

print(f"mean distance to nearest mode — full 200 steps: {full_dist:.3f}   naive 20 steps: {naive_dist:.3f}")

## Practice

These are the hands-on versions of the lesson's practice problems: check the two hand computations in code, then run the four failure modes on purpose and watch the predicted symptoms appear. Try each in the empty cell, then reveal the worked solution.

**Problem 1.** Time embedding by hand. Using only `math.sin` and `math.cos`, compute the fast pair ($\omega_1 = 1$) and the slow pair ($\omega_{16} = 10^{-4}$) of the embedding at $t = 100$, then compare with `emb_all[100]`. Which pair changed more since $t = 50$ (from Step 4's table)?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

```python
fast_sin = math.sin(100)
fast_cos = math.cos(100)
slow_sin = math.sin(100 * 1e-4)
slow_cos = math.cos(100 * 1e-4)

print(f"by hand — fast: ({fast_sin:+.3f}, {fast_cos:+.3f})   slow: ({slow_sin:+.3f}, {slow_cos:+.3f})")
print(f"emb_all  — fast: ({emb_all[100, 0].item():+.3f}, {emb_all[100, 16].item():+.3f})   slow: ({emb_all[100, 15].item():+.3f}, {emb_all[100, 31].item():+.3f})")
```

- Fast pair at $t = 100$: $(\sin 100, \cos 100) = (-0.506, +0.862)$. At $t = 50$ it was $(-0.262, +0.965)$ — a big swing, because the fast hand wrapped about 8 more times between the two.
- Slow pair at $t = 100$: $(\sin 0.01, \cos 0.01) = (+0.010, +1.000)$. At $t = 50$ it was $(+0.005, +1.000)$ — a barely visible creep.

**Answer:** the fast pair changed far more. Fast frequencies separate nearby timesteps; slow frequencies encode global position. The `emb_all` values match the hand values to 3 decimals.

</details>

**Problem 2.** Shape tracking. Reproduce the lesson's shape table in code: run one forward pass piece by piece on a 256-point batch and print the shape after each stage — noisy points, timesteps, embedding, concatenation, output. Confirm the output shape equals the data shape.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

```python
xb = data[:256]
tb = torch.randint(1, T + 1, (256,))
eb = time_embedding(tb)
hb = torch.cat([xb, eb], dim=1)
ob = model(xb, tb)

print(f"noisy points : {tuple(xb.shape)}")
print(f"timesteps    : {tuple(tb.shape)}")
print(f"embedding    : {tuple(eb.shape)}")
print(f"concatenated : {tuple(hb.shape)}")
print(f"output       : {tuple(ob.shape)}")

assert ob.shape == xb.shape
```

**Answer:** $(256, 2) \to (256,) \to (256, 32) \to (256, 34) \to (256, 2)$. The output shape equals the data shape — the network is a data-shaped-in, data-shaped-out denoiser, exactly as the lesson's table predicts.

</details>

**Problem 3.** Learning rate too high. Train a fresh `EpsMLP` with `lr=0.1` (100 times ours) for 800 steps, printing the loss every 100. Predict the symptom before you run, then check.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

```python
model_hot = EpsMLP()
opt_hot = torch.optim.Adam(model_hot.parameters(), lr=0.1)

for step in range(800):
    idx = torch.randint(0, n, (256,))
    x0 = data[idx]
    t = torch.randint(1, T + 1, (256,))
    eps = torch.randn(256, 2)
    xt = sqrt_abar[t - 1][:, None] * x0 + sqrt_1m_abar[t - 1][:, None] * eps
    loss = ((eps - model_hot(xt, t)) ** 2).mean()
    opt_hot.zero_grad()
    loss.backward()
    opt_hot.step()
    if step % 100 == 0:
        print(f"step {step:3d}   loss = {loss.item():.3f}")
```

**Expected outcome:** the loss falls at first but then spikes repeatedly and settles into a jumpy plateau well above the canonical run's 0.25 (with an even larger rate it can hit NaN). Oversized steps overshoot the downhill direction, so the optimizer keeps leaping across the valley instead of descending into it.

**Answer:** learning rate too high — the first row of the lesson's failure-mode table. Fix: return to $10^{-3}$.

</details>

**Problem 4.** $T$ too small. Rebuild the schedule with `T = 20` (same $\beta$ endpoints), retrain a fresh model (1,500 steps is plenty), and sample with the 20-step Algorithm 2. Before running: with $\bar\alpha_{20} = \prod_t (1-\beta_t) \approx 0.82$, how much of the signal is still alive at the end of the forward process, and what will that do to samples started from $\mathcal{N}(0, \mathbf{I})$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

```python
T_small = 20
betas_s = torch.linspace(1e-4, 0.02, T_small)
alphas_s = 1.0 - betas_s
abar_s = torch.cumprod(alphas_s, dim=0)

print(f"abar at T=20: {abar_s[-1].item():.3f}  -> signal amplitude left: {abar_s[-1].sqrt().item():.2f}")

model_s = EpsMLP()
opt_s = torch.optim.Adam(model_s.parameters(), lr=1e-3)
for step in range(1500):
    idx = torch.randint(0, n, (256,))
    x0 = data[idx]
    t = torch.randint(1, T_small + 1, (256,))
    eps = torch.randn(256, 2)
    xt = abar_s[t - 1].sqrt()[:, None] * x0 + (1 - abar_s[t - 1]).sqrt()[:, None] * eps
    loss = ((eps - model_s(xt, t)) ** 2).mean()
    opt_s.zero_grad()
    loss.backward()
    opt_s.step()

with torch.no_grad():
    xt = torch.randn(2000, 2)
    for t in range(T_small, 0, -1):
        tvec = torch.full((2000,), t)
        mean = (xt - betas_s[t - 1] / (1 - abar_s[t - 1]).sqrt() * model_s(xt, tvec)) / alphas_s[t - 1].sqrt()
        if t > 1:
            xt = mean + betas_s[t - 1].sqrt() * torch.randn_like(xt)
        else:
            xt = mean

print(f"mean radius of samples: {xt.norm(dim=1).mean().item():.2f} (target 4.0)")
```

**Expected outcome:** $\bar\alpha_{20} \approx 0.82$, so the forward process ends with $\sqrt{0.82} \approx 0.90$ of the signal amplitude still present — $x_T$ looks nothing like pure noise. But Algorithm 2 starts from $\mathcal{N}(0, \mathbf{I})$ anyway, an input the network never met during training, and the 20-step walk cannot stretch a radius-1.25 cloud to radius 4. The samples huddle at a mean radius around 1.5-2 — far inside the ring — even though the training loss looked perfectly healthy.

**Answer:** $T$ too small — the forward process must genuinely reach noise ($\bar\alpha_T \approx 0$) for the pure-noise starting point to be legitimate.

</details>

**Problem 5.** Forgetting the time embedding. Train a fresh network that ignores $t$ (replace the embedding with zeros in `forward`), 2,500 steps, then sample with the full 200-step Algorithm 2. Predict both the loss plateau and the sample shape before running.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

```python
class EpsMLPNoTime(EpsMLP):
    def forward(self, x, t):
        emb = torch.zeros(x.shape[0], emb_dim)
        h = torch.cat([x, emb], dim=1)
        return self.net(h)

model_nt = EpsMLPNoTime()
opt_nt = torch.optim.Adam(model_nt.parameters(), lr=1e-3)
for step in range(2500):
    idx = torch.randint(0, n, (256,))
    x0 = data[idx]
    t = torch.randint(1, T + 1, (256,))
    eps = torch.randn(256, 2)
    xt = sqrt_abar[t - 1][:, None] * x0 + sqrt_1m_abar[t - 1][:, None] * eps
    loss = ((eps - model_nt(xt, t)) ** 2).mean()
    opt_nt.zero_grad()
    loss.backward()
    opt_nt.step()

print(f"final loss without t: {loss.item():.3f} (canonical run: about 0.25)")

with torch.no_grad():
    xt = torch.randn(2000, 2)
    for t in range(T, 0, -1):
        tvec = torch.full((2000,), t)
        mean = (xt - betas[t - 1] / sqrt_1m_abar[t - 1] * model_nt(xt, tvec)) / torch.sqrt(alphas[t - 1])
        if t > 1:
            xt = mean + torch.sqrt(betas[t - 1]) * torch.randn_like(xt)
        else:
            xt = mean

plt.scatter(xt[:, 0], xt[:, 1], s=2, alpha=0.4, color="#ff7b72")
plt.title("samples without the time embedding")
plt.xlabel("first coordinate")
plt.ylabel("second coordinate")
plt.axis("equal")
plt.show()
```

**Expected outcome:** the loss plateaus noticeably higher than the canonical 0.25, because one compromise guess must serve all 200 noise levels — the network cannot specialize. The samples come out as a smeared ring (or worse), without eight crisp clusters: at every step of the walk the network applies the same average-over-levels correction, which is wrong at almost every individual level.

**Answer:** the missing time embedding — third row of the failure-mode table. The $t$ input is load-bearing.

</details>

**Problem 6.** Sampling without the noise term. Using the **good** trained model, run Algorithm 2 with $z = 0$ at every step (not only the last). Compare the mean distance to the nearest mode center against the canonical 0.19, and describe the picture.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

```python
with torch.no_grad():
    xt = torch.randn(2000, 2)
    for t in range(T, 0, -1):
        tvec = torch.full((2000,), t)
        xt = (xt - betas[t - 1] / sqrt_1m_abar[t - 1] * model(xt, tvec)) / torch.sqrt(alphas[t - 1])

quiet_dist = torch.cdist(xt, centers_8).min(dim=1).values.mean().item()

print(f"mean distance to nearest center with z = 0 everywhere: {quiet_dist:.3f} (canonical: about 0.19)")

plt.scatter(xt[:, 0], xt[:, 1], s=2, alpha=0.4, color="#ff7b72")
plt.title("sampling with z = 0 at every step")
plt.xlabel("first coordinate")
plt.ylabel("second coordinate")
plt.axis("equal")
plt.show()
```

**Expected outcome:** the samples still find the eight modes (the mean update pulls toward high-probability regions), but each cluster collapses to a needle-thin dot: the mean distance to the center drops far below 0.19. The $\sigma_t z$ kicks are the sampler's source of within-cluster diversity — remove them and you remove the spread that the real data has.

**Answer:** dropped sampling noise — fourth row of the failure-mode table. Keep $\sigma_t z$ at every step except the last.

</details>

## Wrap-up

You built and trained a real DDPM and verified every claim along the way: the schedule's endpoints and monotonicity, the time embedding's exact values against hand trigonometry, the 21,250-parameter count, the initial loss of about 1.0, the floor above zero, and samples that hit all the modes at the right radius with the right spread. You also watched naive 20-step skipping fail — each update removes only one step's worth of noise, so skipping 180 of them leaves the job undone. Part 11 reads the trained network a new way — as an estimate of the score, the uphill direction of the log density — and builds DDIM, the sampler that takes 20 big steps correctly.